# 00-3. Preprocessing (U54-DP)

In [26]:
library(data.table)
library(plyr)
library(vespa)
library(viper)
library(reshape2)


Attaching package: ‘reshape2’


The following objects are masked from ‘package:data.table’:

    dcast, melt




In [4]:
if (!dir.exists("./data/u54-dp")) {
  out <- system2(
    "bash",
    "./tools/scripts/u54-dp.sh",
    stdout = TRUE,
    stderr = TRUE
  )

  cat(out, sep = "\n")
}

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100 53.37M 100 53.37M   0      0 130.0M      0                              0
Archive:  ./data.zip
  inflating: ./U54_COAD_phospho.rds  
  inflating: ./U54BL_COAD_phospho.rds  
  inflating: ./U54BL_COAD_proteo.rds  
  inflating: ./U54DP_COAD_phospho.rds  


In [3]:
if (!file.exists("./tools/references/library.fasta")) {
  out <- system2(
    "bash",
    "./tools/scripts/fasta.sh",
    stdout = TRUE,
    stderr = TRUE
  )

  cat(out, sep = "\n")
}

In [12]:
dt <- readRDS("./data/u54-dp/U54DP_COAD_phospho.rds")
dt <- as.data.table(dt)
dt[, run_id := as.character(run_id)]

In [15]:
meta <- unique(dt[, .(run_id)])
meta[, c("cell_code", "drug_code", "time_point") := tstrsplit(run_id, "_", fixed = TRUE)]
meta

run_id,cell_code,drug_code,time_point
<chr>,<chr>,<chr>,<chr>
H508_AL_96h,H508,AL,96h
H508_AL_24h,H508,AL,24h
H508_LI_96h,H508,LI,96h
SNU_OS_24h,SNU,OS,24h
H508_WI_1h,H508,WI,1h
M8_WI_96h,M8,WI,96h
SNU_WI_96h,SNU,WI,96h
LS_RA_24h,LS,RA,24h
LS_C_24h,LS,C,24h


In [16]:
meta[, cell_line := fifelse(cell_code == "H508", "NCI-H508",
                     fifelse(cell_code == "LS",   "LS1034",
                     fifelse(cell_code == "M8",   "MDST8",
                     fifelse(cell_code == "SNU",  "SNU-61",
                     fifelse(cell_code == "H15",  "HCT-15",
                     fifelse(cell_code == "HT",   "HT115", cell_code))))))]

meta[, drug := fifelse(drug_code == "AL", "alpelisib",
                fifelse(drug_code == "IM", "imatinib",
                fifelse(drug_code == "LI", "linsitinib",
                fifelse(drug_code == "OS", "osimertinib",
                fifelse(drug_code == "RA", "ralimetinib",
                fifelse(drug_code == "TR", "trametinib",
                fifelse(drug_code == "WI", "WIKI4",
                fifelse(drug_code == "C",  "DMSO", drug_code))))))))]

In [17]:
dt <- merge(dt, meta, by = "run_id", all.x = TRUE)

In [20]:
get_vpmx<-function(osw, spregulon, aregulon, library) {
  # VESPA (substrate-protein)
  spvpl<-viperize(osw, spregulon, xc=TRUE, normalize=TRUE)
  spvpmx<-reshape2::dcast(spvpl,protein_id~tag,value.var="nes",fill=0)
  spvpmx_proteins<-spvpmx$protein_id
  spvpmx$protein_id<-NULL
  spvpmx<-as.matrix(spvpmx)
  row.names(spvpmx)<-spvpmx_proteins
  
  # VESPA (substrate-protein)
  spvpl2<-viperize(osw, spregulon, xc=TRUE, normalize=FALSE)
  spvpmx2<-reshape2::dcast(spvpl2,protein_id~tag,value.var="nes",fill=0)
  spvpmx2_proteins<-spvpmx2$protein_id
  spvpmx2$protein_id<-NULL
  spvpmx2<-as.matrix(spvpmx2)
  row.names(spvpmx2)<-spvpmx2_proteins
  
  # VESPA (activity-protein)
  apvpl<-viperize(vmx2pv(spvpmx2, fasta=library), aregulon, xc=TRUE, normalize=TRUE)
  apvpmx<-reshape2::dcast(apvpl,protein_id~tag,value.var="nes",fill=0)
  apvpmx_proteins<-apvpmx$protein_id
  apvpmx$protein_id<-NULL
  apvpmx<-as.matrix(apvpmx)
  row.names(apvpmx)<-apvpmx_proteins
  
  # integrate substrate and activity-levels
  ipvpl<-ddply(rbind(melt(spvpmx),melt(apvpmx)),.(Var1,Var2),function(X){data.frame("value"=sum(X$value)/sqrt(length(X$value)))})
  ipvpmx<-dcast(ipvpl, Var1~Var2, value.var="value")
  ipvpmx_proteins<-ipvpmx$Var1
  ipvpmx<-as.matrix(ipvpmx[,-1])
  rownames(ipvpmx)<-ipvpmx_proteins
  
  ipvpmx<-ipvpmx[,colnames(apvpmx)]
  
  return(list("substrate"=spvpmx, "activity"=apvpmx, "integrated"=ipvpmx))
}

In [21]:
viperize<-function(osw, regulon, min_size = 10, topn = 500, xc = TRUE, normalize = TRUE){
  # generate quantitative matrix
  qmn<-export2mx(osw, fillvalues = "colmin")
  
  if (normalize) {
    # rank based normalization of the matrix
    qmn.d1 <- t(t(apply(qmn, 2, rank, na.last = "keep"))/(colSums(!is.na(qmn)) + 1))
    
    # rank based estimation of single sample gene expression signature across the matrix
    qmn.norm <- t(apply(qmn.d1, 1, rank, na.last = "keep"))/(rowSums(!is.na(qmn.d1)) + 1)
  } else {
    qmn.norm <- qmn
  }
  
  # run VIPER
  vpres<-viper(qmn.norm, vespa::pruneRegulon(vespa::subsetRegulon(regulon, rownames(qmn.norm), min_size=min_size), cutoff=topn), minsize=min_size, pleiotropy = xc, pleiotropyArgs = list(regulators = 0.05, shadow = 0.05, targets = 5, penalty = 20, method = "adaptive"), cores=6)
  
  # transform viper results
  vpd<-as.data.frame(vpres)
  vpd$protein_id<-row.names(vpd)
  vpl<-reshape2::melt(vpd, id=c("protein_id"), variable.name = "tag", value.name = "nes")

  return(vpl)
}

In [23]:
library <- "./tools/references/library.fasta"
spregulon <- readRDS("./runs/cptac-coad/vespa.net/results/stdpimeta_substrate_protein_regulon.rds")
aregulon <- readRDS("./runs/cptac-coad/vespa.net/results/dpimeta_activity_protein_regulon.rds")

In [27]:
vpmx_by_cell_line <- lapply(sort(unique(dt$cell_line)), function(cl) {
  message("mVESPA: ", cl)

  get_vpmx(
    osw       = dt[cell_line == cl],
    spregulon = spregulon,
    aregulon  = aregulon,
    library   = library
  )
})

names(vpmx_by_cell_line) <- sort(unique(dt$cell_line))

mVESPA: HCT


Computing the association scores


Computing the association scores

Loading FASTA DB


Computing the association scores

mVESPA: HT115


Computing the association scores


Computing the association scores

Loading FASTA DB


Computing the association scores

mVESPA: LS1034


Computing the association scores


Computing the association scores

Loading FASTA DB


Computing the association scores

mVESPA: MDST8


Computing the association scores


Computing the association scores

Loading FASTA DB


Computing the association scores

mVESPA: NCI-H508


Computing the association scores


Computing the association scores

Loading FASTA DB


Computing the association scores

mVESPA: SNU-61


Computing the association scores


Computing the association scores

Loading FASTA DB


Computing the association scores



In [29]:
dir.create("./data/u54-dp/processed")

In [31]:
saveRDS(vpmx_by_cell_line, "./data/u54-dp/processed/u54-dp_mvespa.rds")